# 모듈 3. 상품 DB 구축

**목표**
1. 한 상품의 **공시 이력**(FWP → 예비본 → 확정본)을 CUSIP으로 묶는다.
2. 공시·조건·기초자산을 분리한 **DuckDB 스키마**를 설계한다.
3. **품질 점검 규칙**으로 추출 오류와 데이터의 특이 사례를 찾아낸다.
4. 모듈 6 분석을 미리 맛본다.

**산출물**: `data/aifinance.duckdb` (표본 DB), `data/aifinance_full.duckdb` (전체 기간 DB, 선택)

## 1. 왜 "공시 = 행" 이 아니라 "상품 = 행" 인가

모듈 1~2에서 발견한 사실들이 DB 설계를 결정합니다.

| 발견 | 설계 결정 |
|---|---|
| 같은 상품의 **예비본과 확정본**이 따로 제출됨 | 문서 테이블과 상품 뷰를 분리. 상품은 **CUSIP**으로 식별 |
| 판매 중 **예비본이 수정 제출**되기도 함 | 첫 예비본(판매 시작)과 마지막 예비본(확정 직전 조건)을 구분 |
| **공시 1건에 노트 여러 종**이 실림 | 문서–CUSIP 연결 테이블(1:N) |
| 기초자산이 상품당 1~5개 | 기초자산 별도 테이블 (기초자산별 집계용) |

```
documents ─┬─< document_cusips >── (CUSIP)
           │
           ├── terms (424B2 1건 = 1행: 추출된 조건)
           │
           └─< underlyings (문서별 기초자산 N행)

products (뷰) = CUSIP 1개 = 상품 1개
              = 확정본 조건 + 예비본 조건 + 판매 기간 + FWP 수
```

DuckDB를 쓰는 이유: 설치가 필요 없는 **파일 하나짜리 분석용 DB**이고, pandas와 바로 주고받으며, SQL이 빠릅니다. 실무에서 엑셀과 대형 DB 사이의 빈자리를 채우기 좋습니다.

In [1]:
import json

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

from aifinance import EdgarClient, html_to_text
from aifinance.config import DATA_DIR, INTERIM_DIR
from aifinance.db import DB_PATH, QUALITY_RULES, build_rows, create_database, find_lineage, run_quality_checks
from aifinance.extract import extract
from aifinance.plotting import setup_korean_font

setup_korean_font()
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 2. 공시 이력 찾기: CUSIP 전문 검색

모듈 1 표본은 무작위라 같은 상품의 예비본과 확정본이 함께 뽑힐 확률이 낮습니다(실제로 7쌍).
그래서 **확정본의 CUSIP으로 EDGAR 전문 검색**을 해서 같은 상품의 모든 공시를 찾습니다. CUSIP은 상품마다 고유하므로 정확한 열쇠입니다.

(결과는 `data/interim/lineage.parquet`에 캐시합니다. 처음 실행하면 약 1분)

In [2]:
sample = pd.read_parquet(INTERIM_DIR / "filings_sample.parquet")
finals = sample[~sample.is_preliminary]
cusips = sorted({extract(r.text, r.accessionNumber).cusip for r in finals.itertuples()} - {None})
print(f"확정본 {len(finals)}건 중 CUSIP 추출 {len(cusips)}건")

client = EdgarClient()
lineage_path = INTERIM_DIR / "lineage.parquet"
if lineage_path.exists():
    lineage = pd.read_parquet(lineage_path)
else:
    lineage = find_lineage(client, cusips)
    lineage.to_parquet(lineage_path)
lineage.form.value_counts()

확정본 150건 중 CUSIP 추출 146건


form
424B2    297
FWP       36
Name: count, dtype: int64

In [3]:
# 상품 하나의 공시 이력
example = lineage.groupby("cusip").size().sort_values().index[-1]
lineage[lineage.cusip == example].sort_values("filingDate")[["cusip", "form", "filingDate", "accessionNumber"]]

,cusip,form,filingDate,accessionNumber
164,46661P5F0,FWP,2026-10-02,0001213900-26-106048
165,46661P5F0,FWP,2026-10-02,0001213900-26-106045
166,46661P5F0,424B2,2026-10-02,0001213900-26-106055
167,46661P5F0,424B2,2026-10-13,0001213900-26-109106


## 3. 추가 예비본 내려받기

이력에서 찾은 424B2 중 표본에 없던 문서(대부분 예비본)를 내려받습니다. FWP는 내려받지 않고 CUSIP과 날짜만 기록합니다.

In [4]:
known = set(sample.accessionNumber)
extra = lineage[(lineage.form == "424B2") & ~lineage.accessionNumber.isin(known)].drop_duplicates("accessionNumber")
extra = extra.assign(text=[html_to_text(client.download(u).read_bytes()) for u in tqdm(extra.url, desc="예비본")])

fwps = lineage[lineage.form == "FWP"].drop_duplicates("accessionNumber")
docs = pd.concat([
    sample[["accessionNumber", "form", "filingDate", "url", "text"]],
    extra[["accessionNumber", "form", "filingDate", "url", "text"]],
    fwps[["accessionNumber", "form", "filingDate", "url", "cusip"]],
], ignore_index=True)
docs.to_parquet(INTERIM_DIR / "documents.parquet")
docs.form.value_counts()

예비본:   0%|          | 0/144 [00:00<?, ?it/s]

예비본:   1%|▏         | 2/144 [00:00<00:08, 17.08it/s]

예비본:   4%|▍         | 6/144 [00:00<00:05, 26.96it/s]

예비본:   6%|▋         | 9/144 [00:00<00:06, 21.46it/s]

예비본:   9%|▉         | 13/144 [00:00<00:04, 26.22it/s]

예비본:  11%|█         | 16/144 [00:00<00:05, 21.92it/s]

... (31 outputs left out)

예비본:  96%|█████████▌| 138/144 [00:05<00:00, 32.68it/s]

예비본:  99%|█████████▊| 142/144 [00:05<00:00, 32.23it/s]

예비본: 100%|██████████| 144/144 [00:05<00:00, 27.67it/s]

form
424B2    444
FWP       36
Name: count, dtype: int64

## 4. DB 만들기

`src/aifinance/db.py`의 두 함수가 일을 나눕니다.
- `build_rows()`: 문서마다 추출기를 돌려 테이블별 DataFrame을 만든다
- `create_database()`: 스키마를 만들고 적재한 뒤 `products` 뷰를 정의한다

In [5]:
rows = build_rows(docs)
con = create_database(rows, DB_PATH)
con.sql('''
    SELECT 'documents' AS 테이블, count(*) AS 행 FROM documents UNION ALL
    SELECT 'document_cusips', count(*) FROM document_cusips UNION ALL
    SELECT 'terms', count(*) FROM terms UNION ALL
    SELECT 'underlyings', count(*) FROM underlyings UNION ALL
    SELECT 'products (뷰)', count(*) FROM products
''').df()

,테이블,행
0,documents,480
1,document_cusips,483
2,terms,444
3,underlyings,707
4,products (뷰),146


In [6]:
con.sql('''
    SELECT cusip, product_family, underlying_names, tenor_years, coupon_rate_pa, prelim_coupon_rate_pa,
           barrier_pct, buffer_pct, estimated_value_per_1000, prelim_filing_date, pricing_date, marketing_days, fwp_count
    FROM products ORDER BY pricing_date DESC LIMIT 10
''').df()

,cusip,product_family,underlying_names,tenor_years,coupon_rate_pa,prelim_coupon_rate_pa,barrier_pct,buffer_pct,estimated_value_per_1000,prelim_filing_date,pricing_date,marketing_days,fwp_count
0,46661Q3F0,issuer_call_contingent_coupon,NDXT / RTY / SPX,1.92,0.109,0.109,0.7,NaN,951.7,2026-11-03,2026-11-06,7,0
1,46661Q2E4,other,SPXFP,2.99,NaN,NaN,NaN,NaN,982.8,2026-11-04,2026-11-06,6,0
2,46661Q2B0,leveraged_buffer,an Unequally Weighted Basket Consisting of the S&P 500 Futures Excess Return...,5.00,NaN,NaN,NaN,0.40,989.1,2026-11-03,2026-11-06,7,0
3,46661QU94,autocall_call_premium,INDU / NDX / RTY,2.99,NaN,NaN,0.7,NaN,953.3,2026-10-27,2026-11-06,14,0
4,46661QKK0,other,NaN,2.17,NaN,NaN,NaN,NaN,970.7,2026-11-03,2026-11-06,7,0
5,46660NFC2,digital,10-Year U.S. Dollar SOFR ICE Swap Rate,NaN,NaN,NaN,NaN,NaN,978.1,2026-11-06,2026-11-05,4,0
6,46661Q6Y6,digital,SPX,1.03,NaN,NaN,0.8,NaN,991.7,2026-11-05,2026-11-05,5,0
7,46661QGK5,autocall_contingent_coupon,Class A Common Stock of Alphabet Inc.,2.00,0.120,NaN,NaN,NaN,960.7,2026-11-02,2026-11-05,7,2
8,46661Q3M5,autocall_call_premium,Class A Common Stock of Alphabet Inc.,2.00,NaN,NaN,NaN,0.10,970.0,2026-11-04,2026-11-03,2,0
9,46661QQM0,dual_directional,SPXFP,1.26,NaN,NaN,NaN,0.15,986.5,2026-10-22,2026-11-03,14,0


## 5. 품질 점검

추출기는 완벽하지 않습니다. 대량 데이터에서는 **규칙으로 이상치를 걸러내는 것**이 문서를 다시 읽는 것보다 효율적입니다.
규칙은 세 등급으로 나눕니다.
- **error**: 분석에 쓰면 안 되는 행 (필수값 누락, 날짜 순서 오류)
- **warning**: 추출 오류일 가능성이 큰 행 (범위 밖 값, 상품군과 조건 불일치)
- **info**: 오류는 아니지만 알아야 할 특이 사례 (노트 여러 종, 예비본 수정)

In [7]:
pd.DataFrame(QUALITY_RULES, columns=["rule_id", "severity", "rule", "sql"])[["rule_id", "severity", "rule"]]

,rule_id,severity,rule
0,Q01,error,확정본에 CUSIP 없음
1,Q02,error,확정본에 가격결정일·만기일 없음
2,Q03,error,날짜 순서 오류 (가격결정일 ≤ 발행일 < 만기일)
3,Q04,warning,비율 값이 상식 범위 밖
4,Q05,warning,"추정가치가 $850~$1,000 범위 밖"
5,Q06,warning,상품군과 핵심 조건 불일치
6,Q07,warning,상품군 미분류 (other)
7,Q08,info,공시 1건에 CUSIP 여러 개 (노트 여러 종)
8,Q09,warning,같은 CUSIP의 확정본이 여러 건
9,Q10,warning,확정 쿠폰이 예비본 하한보다 낮음


In [8]:
summary = run_quality_checks(con)
summary

,rule_id,severity,rule,n_issues
0,Q01,error,확정본에 CUSIP 없음,4
1,Q02,error,확정본에 가격결정일·만기일 없음,1
2,Q03,error,날짜 순서 오류 (가격결정일 ≤ 발행일 < 만기일),0
3,Q04,warning,비율 값이 상식 범위 밖,0
4,Q05,warning,"추정가치가 $850~$1,000 범위 밖",0
5,Q06,warning,상품군과 핵심 조건 불일치,15
6,Q07,warning,상품군 미분류 (other),20
7,Q08,info,공시 1건에 CUSIP 여러 개 (노트 여러 종),1
8,Q09,warning,같은 CUSIP의 확정본이 여러 건,0
9,Q10,warning,확정 쿠폰이 예비본 하한보다 낮음,0


In [9]:
con.sql("SELECT rule_id, accession_number, cusip, detail FROM quality_issues WHERE severity <> 'info' OR rule_id = 'Q12' ORDER BY rule_id").df()

,rule_id,accession_number,cusip,detail
0,Q01,0001213900-26-096607,NaN,"(EEM Notes) Capped Buffered Return Enhanced Notes due September 6, 2028"
1,Q01,0001213900-26-110046,NaN,
2,Q01,0001213900-26-105084,NaN,
3,Q01,0001213900-26-095682,NaN,
4,Q02,0001213900-26-117911,46660NFC2,pricing=2026-11-05 maturity=
5,Q06,0001213900-26-109106,46661P5F0,autocall_contingent_coupon
6,Q06,0001918704-26-025022,46661MEX8,leveraged_barrier
7,Q06,0001918704-26-033413,46661QS30,leveraged_barrier
8,Q06,0001918704-26-028332,46661PYA9,autocall_contingent_coupon
9,Q06,0001213900-26-107226,46661PH75,autocall_contingent_coupon


### 품질 점검이 찾아낸 것

이 규칙들을 처음 돌렸을 때 나온 결과를 하나씩 원문과 대조해 고친 기록입니다. **품질 점검 → 원인 조사 → 추출기 수정**의 반복이 실무 데이터 엔지니어링의 핵심입니다.

| 규칙 | 처음 건수 | 원인 | 조치 |
|---|---|---|---|
| Q05 추정가치 범위 밖 | 3 | UBS 형식은 액면이 **$10** (`$9.901 per $10`) | 액면을 읽어 $1,000 기준으로 환산 |
| Q02 만기일 누락 | 13 | 항목명 뒤 **각주 기호** (`Maturity Date † :`, `Stated Maturity Date 1 :`) | 구분자 패턴에 각주 허용 |
| Q10 확정 쿠폰 < 예비본 하한 | 1 | 추출 오류가 아니었음. 판매 중 **예비본이 수정 제출**되며 하한이 14.00% → 12.50%로 내려감 | 첫/마지막 예비본을 구분, Q12 규칙 신설 |
| Q08 노트 여러 종 | 1 | 공시 1건에 기초자산별 노트 6종 | 실습 3-B에서 상품 분리 |
| Q07 미분류 | 20 | UBS·Goldman 플랫폼 형식, 고정금리 콜러블 등 | 실습 2-C (모듈 2) |

> Q10은 좋은 교훈입니다. **"이상치 = 추출 오류"라고 단정하지 말고 원문을 확인해야** 합니다. 이 경우 이상치가 시장 상황에 따른 조건 조정이라는 실제 사건이었습니다.

## 6. 미리 보는 분석

DB가 있으면 질문 하나가 SQL 몇 줄입니다. 모듈 6에서 전체 기간 데이터로 본격적으로 다룰 질문들을 미리 던져 봅니다.
(표본 146개 상품 기준이므로 숫자는 참고용입니다.)

In [10]:
# Q1. 상품군별 발행 건수·금액, 평균 만기, 평균 추정가치
con.sql('''
    SELECT product_family,
           count(*)                                      AS 상품수,
           round(sum(principal_amount_usd) / 1e6, 1)     AS 발행금액_백만달러,
           round(median(principal_amount_usd) / 1e3, 0) AS 중앙값_천달러,
           round(avg(tenor_years), 1)                    AS 평균만기_년,
           round(avg(estimated_value_per_1000), 1)       AS 평균추정가치
    FROM products GROUP BY ALL ORDER BY 상품수 DESC
''').df()

,product_family,상품수,발행금액_백만달러,중앙값_천달러,평균만기_년,평균추정가치
0,autocall_contingent_coupon,43,106.4,995.0,3.8,942.8
1,autocall_call_premium,24,47.4,996.0,4.1,939.0
2,issuer_call_contingent_coupon,18,42.8,743.0,3.5,955.0
3,other,17,245.6,2393.0,2.2,978.3
4,digital,12,44.2,1000.0,1.8,982.3
5,dual_directional,11,18.9,1540.0,2.1,980.9
6,leveraged_buffer,11,15.4,600.0,3.0,979.1
7,leveraged_barrier,10,10.6,1050.0,4.5,968.3


> **발행사 마진의 대리변수**: 추정가치는 JPM이 내부 모형으로 계산한 상품 가치입니다. 발행가 $1,000과의 차이(약 3~6%)에 판매 수수료, 헤지 비용, 발행사 이익이 들어 있습니다. 상품군별로 이 차이가 어떻게 다른지가 모듈 6의 핵심 질문 중 하나입니다.

In [11]:
# Q2. 예비본의 '최소' 쿠폰 대비 확정 쿠폰은 어떻게 정해지나
uplift = con.sql('''
    SELECT product_family,
           count(*) AS n,
           count_if(abs(coupon_rate_pa - prelim_coupon_rate_pa) < 1e-9) AS 하한_그대로,
           count_if(coupon_rate_pa > prelim_coupon_rate_pa + 1e-9)       AS 하한_초과,
           round(avg((coupon_rate_pa - prelim_coupon_rate_pa) * 1e4), 1)  AS 평균차이_bp
    FROM products WHERE coupon_rate_pa IS NOT NULL AND prelim_coupon_rate_pa IS NOT NULL
    GROUP BY ALL ORDER BY n DESC
''').df()
uplift

,product_family,n,하한_그대로,하한_초과,평균차이_bp
0,autocall_contingent_coupon,29,29.0,0.0,0.0
1,issuer_call_contingent_coupon,18,17.0,1.0,2.8
2,other,3,3.0,0.0,0.0


> 확정 쿠폰은 거의 항상 **예비본의 "최소" 쿠폰과 같습니다.** 판매 기간 중 시장이 유리하게 움직여도 그 이익이 투자자 쿠폰으로 가지 않는다는 뜻일 수 있습니다. 그렇다면 그 차이는 어디로 갈까요? (힌트: 추정가치. 모듈 6에서 확인합니다.)

In [12]:
# Q3. 판매 기간: 첫 예비본 제출 → 확정본 제출
con.sql('''
    SELECT product_family, count(*) AS n, round(avg(marketing_days), 1) AS 평균일, min(marketing_days) AS 최소, max(marketing_days) AS 최대
    FROM products WHERE marketing_days IS NOT NULL GROUP BY ALL ORDER BY n DESC
''').df()

,product_family,n,평균일,최소,최대
0,autocall_contingent_coupon,43,11.8,1,33
1,autocall_call_premium,24,13.1,2,32
2,issuer_call_contingent_coupon,18,10.1,4,18
3,other,17,8.2,1,17
4,digital,12,10.5,2,26
5,dual_directional,11,13.9,4,31
6,leveraged_buffer,11,8.9,2,28
7,leveraged_barrier,10,9.7,2,25


In [13]:
# Q4. 기초자산 Top 15 (확정본 기준)
con.sql('''
    SELECT coalesce(u.ticker, u.name) AS 기초자산, u.kind, count(*) AS 상품수
    FROM products p JOIN underlyings u ON u.accession_number = p.final_accession
    GROUP BY ALL ORDER BY 상품수 DESC LIMIT 15
''').df()

,기초자산,kind,상품수
0,SPX,index,48
1,RTY,index,42
2,NDX,index,22
3,MQUSLVA,index,17
4,SPXFP,index,12
5,MQUSTVA,index,11
6,INDU,index,10
7,NDXT,index,7
8,SX5E,index,7
9,Class A Common Stock of Alphabet Inc.,stock,4


## 7. 전체 기간으로 확장하기 (선택)

표본 대신 기간 전체를 수집해 DB를 만드는 명령입니다. 모듈 6은 이 DB를 씁니다.

```bash
uv run python -m aifinance.pipeline --start 2026-07-29 --end 2026-11-10
```

- 약 100일, 424B2·FWP 6~7천 건, 30~40분 (SEC 요청 속도 제한 때문)
- 예비본이 확정본보다 1~3주 먼저 나오므로 **분석 기간보다 2주 앞서** 시작
- 전체 기간에서는 전문 검색 없이 **수집한 문서끼리 CUSIP으로 바로 연결**됩니다

In [14]:
full_db = DATA_DIR / "aifinance_full.duckdb"
if full_db.exists():
    full = duckdb.connect(str(full_db), read_only=True)
    display(full.sql("SELECT count(*) AS 상품수, count(prelim_accession) AS 예비본_연결, min(pricing_date) AS 시작, max(pricing_date) AS 끝 FROM products").df())
    full.close()
else:
    print("전체 기간 DB가 아직 없습니다. 위 명령으로 만들 수 있습니다.")

,상품수,예비본_연결,시작,끝
0,2971,2629,2026-06-25,2026-11-09


## 🤖 Claude 실습 3

**3-A. 말로 묻고 SQL로 답 받기 (Text-to-SQL)**
> `data/aifinance.duckdb`의 테이블과 `products` 뷰 구조를 먼저 확인한 뒤, 다음 질문에 답하는 SQL을 작성하고 실행해줘. 쿼리와 결과를 함께 보여주고, 결과를 실무자 관점에서 한두 문장으로 해석해줘.
> 1. 워스트오브(worst_of) 상품은 단일 기초자산 상품보다 쿠폰이 얼마나 높은가? 상품군별로.
> 2. 배리어가 낮을수록(더 안전할수록) 쿠폰은 얼마나 낮아지는가?
> 3. FWP를 낸 상품과 안 낸 상품은 발행금액이 다른가?

**3-B. 노트 여러 종이 실린 공시 분리**
> 품질 점검 Q08에 걸린 공시는 노트 여러 종을 한 문서에 담고 있어. 원문을 읽고 노트별 CUSIP·기초자산·발행금액·상한·추정가치를 표로 정리해줘. 그다음 이런 문서를 상품별 행으로 쪼개는 함수를 `src/aifinance/extract.py`에 추가하는 방법을 제안하고 구현해줘.

**3-C. 품질 규칙 추가**
> `src/aifinance/db.py`의 `QUALITY_RULES`에 규칙 3개를 추가해줘. 예: 만기가 비정상적으로 짧거나 긴 상품, 조기상환 빈도와 쿠폰 빈도가 다른 오토콜 상품, 쿠폰 배리어가 만기 배리어보다 낮은 상품. 각 규칙에 걸린 사례를 원문과 대조해 진짜 이상치인지, 추출 오류인지, 정상적인 상품 설계인지 판정해줘.

**3-D. 예비본 수정 사례 조사** (전체 기간 DB가 있으면)
> `data/aifinance_full.duckdb`에서 판매 중 예비본이 수정된 상품(Q12)을 모두 찾아, 쿠폰 하한이 어떻게 바뀌었는지와 그 기간의 시장 상황(기초자산 움직임)을 연결해 설명해줘.

---
**다음 모듈**: 모듈 4에서는 공시 원문을 청크로 나눠 로컬 임베딩으로 벡터화하고, 상품 DB와 벡터 검색을 **MCP 서버**로 Claude Desktop에 연결합니다.